# 06 — Choose a model, then choose an action threshold

**Plain-language question:** Is choosing the model the same as choosing when it
should say “yes”?

**Why this matters:** model selection compares ranking quality; threshold
selection decides who receives an action. Mixing the two makes evaluation hard
to reason about and easy to bias.

**Estimated time:** 60–75 minutes.
**Prerequisite:** lessons 00–05; you understand probabilities, confusion counts,
the baseline, and a fitted sklearn Pipeline.


## Preflight

Check the kernel, then rebuild the visible development objects.


In [ ]:
import importlib.util
import sys

required = ("mlflow", "pandas", "sklearn", "aai_local_classification")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        "This notebook is using the wrong Python kernel. Close this Jupyter "
        "server, run `make notebook` from examples/local-classification, or "
        "select the 'AAI Local Classification' kernel. Missing: " + ", ".join(missing)
    )

import pandas as pd

from aai_local_classification.learning import study_root
from aai_local_classification.settings import load_settings
from aai_local_classification.tracking import local_paths

settings = load_settings()
root = study_root()
paths = local_paths(root)
print(f"✓ Python {sys.version_info.major}.{sys.version_info.minor}: {sys.executable}")
print("✓ Course imports are available")
print(f"✓ Learner state: {root}")


In [ ]:
from sklearn.metrics import average_precision_score

from aai_local_classification.contracts import SplitName
from aai_local_classification.data import load_split
from aai_local_classification.evaluation import (
    evaluate_probabilities,
    select_threshold,
)
from aai_local_classification.learning import state_exists
from aai_local_classification.modeling import (
    build_candidate,
    candidate_specs,
    feature_frame,
)
from aai_local_classification.workflow import (
    ensure_prepared,
    get_or_run_candidate_selection,
)


The import cell names the reusable operations; it has not fit a model. Now load
only train and validation.


In [ ]:
ensure_prepared(settings, root)
train = load_split(settings, SplitName.TRAIN, paths.data_root)
validation = load_split(settings, SplitName.VALIDATION, paths.data_root)
X_train = feature_frame(train, settings)
y_train = train.churned_30d
X_validation = feature_frame(validation, settings)
y_validation = validation.churned_30d
print("✓ Development data ready; test remains unopened")


### What you should see

A success message confirming that the test is still unopened.

### Words introduced

| Word | Plain meaning | Here |
|---|---|---|
| ranking | Ordering rows from lower to higher model score | AP evaluates it |
| candidate | One declared model specification being compared | logistic or forest |
| fair comparison | Hold data and evaluation rule constant | same train/validation |


## Fit both candidates on exactly the same rows

The two candidates use the same preprocessing and random seed. Their classifier
families are the controlled change.

**Before you run this:** the more complex random forest may score slightly
higher. Predict whether “slightly higher” must always win.


In [ ]:
manual_models = {}
manual_probabilities = {}
comparison_records = []

for spec in candidate_specs():
    model = build_candidate(spec, settings).fit(X_train, y_train)
    positive_index = list(model.classes_).index(1)
    probability = model.predict_proba(X_validation)[:, positive_index]
    average_precision = average_precision_score(y_validation, probability)
    manual_models[spec.name] = model
    manual_probabilities[spec.name] = probability
    comparison_records.append(
        {
            "candidate": spec.name,
            "complexity_rank": spec.complexity_rank,
            "validation_average_precision": average_precision,
        }
    )

manual_comparison = pd.DataFrame(comparison_records)
manual_comparison


### What you should see

Logistic regression AP is about 0.466 and random forest AP about 0.475. Both
rank much better than the roughly 0.194 no-skill baseline. The difference
between the candidates is only about 0.009.

### How to interpret the output

AP ignores a particular threshold and evaluates score ranking across possible
operating points. Higher is better, but tiny differences may not justify a more
complex model.


## Apply the declared simplicity tolerance visibly

The policy says: find the best validation AP, then consider a simpler candidate
eligible when it is within `0.02` of that best score. Among eligible candidates,
prefer lower complexity.


In [ ]:
best_ap = manual_comparison.validation_average_precision.max()
manual_comparison["gap_from_best"] = (
    best_ap - manual_comparison.validation_average_precision
)
manual_comparison["within_tolerance"] = (
    manual_comparison.gap_from_best <= settings.selection.simpler_model_tolerance
)
eligible_manual = manual_comparison.loc[manual_comparison.within_tolerance]
manual_selected_name = eligible_manual.sort_values("complexity_rank").iloc[0].candidate
selected_probability = manual_probabilities[manual_selected_name]
manual_comparison


### What you should see

Both rows are within `0.02`; the selected name is
`logistic-regression` because it has complexity rank 1. This is a predeclared
preference, not a story invented after seeing test data.

### Misconception check

“More complex” does not mean “more production-ready.” Complexity can increase
maintenance and explanation cost. A different project may declare a different
tolerance or no simplicity preference at all.


## A model score is not yet an action

### Words introduced

| Word | Plain meaning | Example |
|---|---|---|
| threshold | Score at or above which prediction becomes 1 | `0.12` |
| operating point | Precision/recall/workload at one threshold | one table row |
| constraint | A minimum/maximum an option must satisfy | recall at least 0.75 |

For a score of 0.20, threshold 0.12 says positive while threshold 0.50 says
negative. The fitted model and its AP have not changed—only the action rule has.

The teaching **cost per 1,000** is
`(5 × false negatives + 1 × false positives) / row count × 1,000`. The cost
units are fictional; the calculation makes an authored trade-off comparable
across batches of different sizes.


**Before you run this:** as the threshold rises, predict what generally happens
to the number of positive predictions and to recall.


In [ ]:
def threshold_record(threshold):
    result = evaluate_probabilities(
        y_validation,
        selected_probability,
        threshold,
        false_negative_cost=settings.selection.false_negative_cost,
        false_positive_cost=settings.selection.false_positive_cost,
    )
    feasible = (
        result.precision >= settings.selection.minimum_validation_precision
        and result.recall >= settings.selection.minimum_validation_recall
    )
    return {
        "threshold": threshold,
        "precision": result.precision,
        "recall": result.recall,
        "predicted_positive_rate": result.predicted_positive_rate,
        "cost_per_1000": result.cost_per_1000,
        "feasible": feasible,
    }


The helper above evaluates one visible threshold; it does not choose anything.
Now apply it to five illustrative operating points.


In [ ]:
sample_thresholds = [0.10, 0.12, 0.13, 0.20, 0.50]
threshold_table = pd.DataFrame(threshold_record(value) for value in sample_thresholds)
threshold_table


### How to interpret the output

At 0.50, recall is very low. At 0.12, recall is about 0.829 and precision
about 0.305; roughly 53% of rows receive a positive action. The lower threshold
accepts more false positives to avoid expensive false negatives.

The exact policy searches a finer grid, keeps rows meeting minimum precision
0.30 and recall 0.75, then chooses minimum teaching cost.


In [ ]:
manual_threshold = select_threshold(
    y_validation,
    selected_probability,
    settings,
)
pd.Series(
    {
        "chosen_threshold": manual_threshold.threshold,
        "precision": manual_threshold.validation_metrics.precision,
        "recall": manual_threshold.validation_metrics.recall,
        "cost_per_1000": manual_threshold.validation_metrics.cost_per_1000,
        "feasible_thresholds": manual_threshold.feasible_threshold_count,
    }
).to_frame("validation result")


### What you should see

Threshold `0.12`, precision about 0.305, recall about 0.829, and cost about
533.3 per 1,000. If no threshold met both constraints, selection would stop as
inconclusive before touching test data.


In [ ]:
import matplotlib.pyplot as plt

plot_table = threshold_table.set_index("threshold")
ax = plot_table[["precision", "recall", "predicted_positive_rate"]].plot(
    marker="o", figsize=(8, 4), title="Validation trade-offs at sample thresholds"
)
ax.axhline(
    settings.selection.minimum_validation_precision, linestyle="--", color="gray"
)
ax.axhline(settings.selection.minimum_validation_recall, linestyle=":", color="gray")
ax.set_ylabel("share")
plt.tight_layout()


The chart makes the trade-off visible. Dashed guide lines are constraints, not
universal standards. The action owner must also consider review capacity and
whether the assumed costs are defensible.

## Record the controlled comparison in MLflow

Only now do we call the packaged helper. It repeats the operations just shown,
logs each candidate as a nested run, stores each complete Pipeline, and persists
the selection evidence. If compatible evidence already exists, it reuses it so
a later frozen-test decision cannot be disconnected from its chosen model.


In [ ]:
selection_already_existed = state_exists("selection.json")
selection = get_or_run_candidate_selection(settings, root)
chosen = next(
    item for item in selection.candidates if item.run_id == selection.selected_run_id
)
print(
    "Reused compatible selection" if selection_already_existed else "Created selection"
)
pd.Series(
    {
        "selected_candidate": selection.selected_candidate,
        "threshold": chosen.threshold_selection.threshold,
        "validation_average_precision": chosen.threshold_selection.validation_metrics.average_precision,
        "test_data_accessed": False,
    }
).to_frame("recorded value")


### What you should see

The recorded candidate and threshold match the manual result, and
`test_data_accessed` remains false. The helper also logged parameters, metrics,
dataset inputs, model signature, input example, configuration, manifest, and
dependency lock.


In [ ]:
import mlflow

mlflow.set_tracking_uri(paths.tracking_uri)
client = mlflow.MlflowClient()
recorded_run = client.get_run(selection.selected_run_id)
pd.Series(
    {
        "run_name": recorded_run.data.tags.get("mlflow.runName"),
        "model_family": recorded_run.data.params.get("model_family"),
        "validation_average_precision": recorded_run.data.metrics.get(
            "validation_average_precision"
        ),
        "artifact_names": [
            item.path for item in client.list_artifacts(selection.selected_run_id)
        ],
    }
).to_frame("MLflow value")


### MLOps bridge

The MLflow run makes the comparison auditable: same data inputs, explicit
parameters, linked metrics, a loadable model, and reproducibility artifacts.
The model's restore environment uses the exact runtime dependency closure
exported from `uv.lock`, not only a few top-level package pins.
Run `make mlflow-ui` in a second Terminal to explore the same local experiment;
stop that server with Ctrl-C.


### Guided exercise

In a copied setting, double the false-positive cost from 1 to 2 and rerun only
threshold selection on the already-computed validation probabilities. This is
safe scratch work; it does not replace the persisted selection.


In [ ]:
exercise_selection_policy = settings.selection.model_copy(
    update={"false_positive_cost": 2.0}
)
exercise_settings = settings.model_copy(update={"selection": exercise_selection_policy})
exercise_threshold = select_threshold(
    y_validation,
    selected_probability,
    exercise_settings,
)
pd.Series(
    {
        "original_threshold": manual_threshold.threshold,
        "new_threshold": exercise_threshold.threshold,
        "original_precision": manual_threshold.validation_metrics.precision,
        "new_precision": exercise_threshold.validation_metrics.precision,
    }
)


**Self-check:** making false alarms more expensive raises the chosen threshold
in this deterministic exercise. It trades some recall for higher precision.
Discrete thresholds and constraints mean the exact move is a policy result,
not a universal mathematical guarantee.

<details><summary>Solution explanation</summary>

The selected threshold moves from 0.12 to 0.14. Precision rises from about
0.305 to about 0.321 while recall falls from about 0.829 to about 0.757. Raising
the threshold flags fewer accounts when false alarms cost more.
</details>


In [ ]:
# Reference solution — run after your attempt
assert abs(exercise_threshold.threshold - 0.14) < 1e-9
assert (
    exercise_threshold.validation_metrics.precision
    >= manual_threshold.validation_metrics.precision
)
print("✓ Higher false-positive cost produced a higher-threshold policy")


## Recap

- Validation AP chooses ranking behavior; it does not choose an action threshold.
- A predeclared tolerance selects the simpler logistic candidate here.
- Constraints and error costs choose threshold 0.12 on validation only.

**Evidence created:** MLflow selection and candidate runs plus `selection.json`.
A compatible rerun reuses this selection once later test evidence exists.

**Ready for 07?** You can explain why AP stays fixed when only the threshold
changes and why test data has not yet been opened.
